In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, glob, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0   = 20260803
IDX_S   = 1
NB_BOOT = 4000

# ── 창 (R = index 100 · 360Hz)
STT_SEG  = (130, 215)       # Q7-F/I/I″/I‴ 승계 — R+30~+319ms
STT_SAFE = (130, 175)       # ★ R+30~+208ms. post_rr < 75샘플이어야 침입 → 사실상 불가
INTRUDE_LIM = STT_SEG[1] - 100      # 이 값보다 post_rr 이 짧으면 다음 QRS 가 창에 든다
# ★ Q7-F 의 P 창 3종을 **정확 정합 조건에서 다시 잰다** — 지금까지 나온 「형태 상한
#   0.6890」은 **절대** pre_rr 대역 정합값이라 Q7-I‴ K3 가 무효화했다. 여기서 처음으로
#   오염되지 않은 형태 단독 값이 나온다.
SEGS = {"p_full": (0, 85), "p_early": (0, 32), "p_late": (53, 85),
        "stt": STT_SEG, "stt_safe": STT_SAFE}          # Q7-F 승계

# ── 사전등록 상수. **이 아래 어느 셀에서도 다시 고르지 않는다.**
KS         = (8, 16)        # Q7-I 승계 — 국소 기저선 창
LB_K       = 16             # ★ 정합에 쓰는 국소 기저선 (f2_16 과 짝)
MIN_S_TPL  = 20             # Q7-F/H/I/I‴ 승계
K_FOLD     = 5              # R22
N_REPEAT   = 3
N_SHUF     = 20             # ★ 셔플 null 반복. 5회로는 작은 초과분(±0.02)을
                            #   판정 못 한다 → null 의 셔플 오차를 CI 에 전파한다
LB_BANDS   = (0.10, 0.06, 0.04, 0.02)   # ★ 국소 기저선 대역(중앙 RR 대비) — 곡선
MIN_CELL_S, MIN_CELL_N = 3, 3
MIN_JOINT_S, MIN_JOINT_PAIR = 15, 100
MIN_JOINT_REC = 10
EXACT_TIE   = 0.505         # (참조) 정확 pre 정합이면 f1 은 동점뿐 — 곡선 첫 줄
F1_COLLAPSE = 0.52          # L1 — 합동 정합은 **두 축 모두 대역**이라 동점이 아니다
F2_COLLAPSE = 0.56          # ★ L2 — 이 아래여야 「조기성을 통제했다」
BONF3       = 0.05 / 3 / 2  # ★ 1차 가족은 **{L3 형태 · L4 post_rr · L7 P창}** 셋뿐.
                            #   L5·L6·L8 은 **민감도 분석**이라 미보정으로 내고 그렇게 적는다
ISO_HI, ISO_LO = 0.7, 0.3   # Q7-F/I 승계

CONFIG = dict(
    exp="quest46_q7k_relative_match", quest="ailab-2026-0046", step="svdb-relative-match",
    parent_exp=["quest46_q7i3_match_exact", "ailab-2026-0060"],
    purpose=("Q7-I‴ K3 이 문을 열었다 — `pre_rr` 을 **값 단위로** 맞춰도 `f2`(국소 기저선 "
             "대비 조기성)가 0.6688 로 살아남는다. **절대 RR 정합은 조기성의 눈금만 맞추고 "
             "상대성은 손대지 않았다.** 그래서 이 퀘스트가 지금껏 「정합 후」라고 부른 값들은 "
             "조기성을 다 뺀 값이 아니다. 여기서는 **`pre_rr` 값 × 국소 기저선 대역**으로 "
             "**합동 정합**해 `f1` 과 `f2` 를 **둘 다** 무너뜨리고, 그 조건에서 **`STT`(형태)와 "
             "`f3`(post_rr)가 남는지**를 묻는다. 이 질문은 아직 한 번도 물어본 적이 없다"),
    dataset="SVDB 전수 · svdb_data5.npz + Q7-B 예측 캐시(라벨·매핑용)",
    lb_bands=list(LB_BANDS), ks=list(KS), lb_k=LB_K, n_shuffle=N_SHUF,
    windows={k: list(v) for k, v in SEGS.items()}, intrude_lim=INTRUDE_LIM,
    predictions={
        "L0": "(관문 아님) 국소 기저선 격자 · 대역별 실효 폭 · **짝 단위 잔여를 f1·f2 둘 다**",
        "L1": f"합동 정합에서 f1ₘ CI 상한 < {F1_COLLAPSE} — 절대 조기성이 죽었나",
        "L2": f"★ 합동 정합에서 f2_16ₘ CI 상한 < {F2_COLLAPSE} (Q7-I‴ 은 0.6688) — **여기가 지지여야 "
              "아래를 「조기성 통제 후」로 읽는다**",
        "L3": "★ STTₘ − nullSTTₘ CI 하한 > 0 (Bonferroni 3) — **형태의 진짜 독립 기여**",
        "L4": "f3ₘ − 0.5 CI 하한 > 0 (Bonferroni 3) — **post_rr 의 진짜 독립 기여**",
        "L5": "(STT 초과) − (f3 초과) (Bonferroni 3) — 둘 중 누가 큰가",
        "L6": "STT_SAFEₘ − nullₘ CI 하한 > 0 — **다음 QRS 침입을 뺀 짧은 창**에서도 남나",
        "L7": "★ P_lateₘ − nullₘ CI 하한 > 0 (Bonferroni 3) — **Q7-F 의 P 창을 "
              "오염되지 않은 조건에서 처음 잰다**. 참고로 P_full 병기",
        "L8": "(민감도) `post_rr` 까지 정합했을 때 STT 초과가 남나 — 남으면 재분극의 "
              "rate hysteresis 경로가 아니다",
        "L9": "(관문 아님) ★ **정합 코호트의 선택 편향** · 폴드 간 계수 부호 안정성 · "
              f"post_rr < {INTRUDE_LIM}샘플 침입률 · 비트 수준 층화 일치도"},
    caveat=("**L2 가 지지가 아니면 L3~L6 을 「조기성 통제 후」로 인용하지 않는다**(Q7-I″ 의 "
            "봉인 규약 승계 · R24-b). 유효 대역은 **규칙이 고른다**(정합 가능 개체 ≥ "
            f"{MIN_JOINT_REC} 중 가장 좁은 것) — 데이터를 보고 고르지 않는다. **학습이 낀 팔은 "
            "전부 자기 라벨셔플 null 위 초과분**으로 읽고, **null 의 셔플 간 오차를 CI 에 "
            "전파**한다. 순수 특징(f1·f2·f3)의 영분포는 0.5 다. 개체 내부 로지스틱·템플릿은 "
            "**라벨을 쓰는 상한**이지 배포 모형이 아니다(Q7-B′ 의 0.8842 와 직접 비교 금지). "
            "★ **정확/합동 정합은 코호트를 고른다** — RR 분포가 겹치는(=조기성이 덜 뚜렷한) "
            "개체가 남는다. **L9 에서 층 구성을 반드시 보고**하고, 런 우세 층이 0 이면 "
            "**결론의 적용 범위를 「고립 S 위주」로 한정**해 적는다. ★ **미리 적어둔다 — "
            "개체당 비트가 ~2000 이라 두 축을 동시에 좁히면 칸이 잘게 쪼개져 L2 가 미결로 "
            "나올 수 있다.** 그러면 봉인이 정상 작동한 것이고, 결론은 「형태가 없다」가 아니라 "
            "**「이 데이터로는 상대 조기성까지 통제할 수 없다」**다 — 그렇게 적는다. 학습 0회. "
            "**매크로가 주지표**이고 비트 수준 층화 일치도는 검정력 보조다(사전 선언)"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7k_relative_match", CONFIG, project=PROJECT)
run.log("설정 ✅ 합동 정합(pre_rr 값 × 국소기저선 대역) · 대역 " + str(LB_BANDS))

In [ ]:
# CELL 2 — 【L-0a】 자산 · 매핑 (Q7-D/E/F/H/I/I″/I‴ 와 동일 규약 — fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")
labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
POST = np.asarray(d5["post_rr"])[keep].astype(float)
BEAT = np.asarray(d5["beat"])[keep]
WIN = {k: np.ascontiguousarray(BEAT[:, :, a:b]).astype("float32")
       for k, (a, b) in SEGS.items()}
del BEAT
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【L-0a】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(ALLR)}개 · "
        "창 " + " · ".join(f"{k}{v} 폭{v[1]-v[0]}" for k, v in SEGS.items()))
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【L-A】 특징·점수 (Q7-I/I‴ 승계 · 여기서 재설계하지 않는다)
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

def local_base(pre_v, k):
    n = len(pre_v); out = np.empty(n); med = float(np.median(pre_v))
    for i in range(n):
        a = max(0, i - k)
        out[i] = med if i - a < 3 else float(np.median(pre_v[a:i]))
    return out

def rhythm_feats(pre_v, post_v, ks):
    """Q7-I‴ 승계. **국소 기저선(LB_K)을 함께 돌려준다 — 정합 키의 두 번째 축이다.**"""
    med = float(np.median(pre_v)); F, NM = [], []
    F.append(med - pre_v); NM.append("f1")
    b_first = None; b_lb = None
    for k in ks:
        b = local_base(pre_v, k)
        if b_first is None:
            b_first = b
        if k == LB_K:
            b_lb = b
        F.append(1.0 - pre_v / np.maximum(b, 1e-9)); NM.append(f"f2_{k}")
    F.append(1.0 - (pre_v + post_v) / np.maximum(2.0 * b_first, 1e-9)); NM.append("f3")
    cv = np.empty(len(pre_v))
    for i in range(len(pre_v)):
        a = max(0, i - ks[0]); w = pre_v[a:i] if i - a >= 3 else pre_v[:3]
        cv[i] = float(np.std(w) / max(np.mean(w), 1e-9))
    F.append(cv); NM.append("f4")
    F.append(np.r_[0.0, F[1][:-1]]); NM.append("f5")
    F.append(1.0 - b_lb / max(med, 1e-9)); NM.append("f6")
    if b_lb is None:
        raise AssetError(f"LB_K={LB_K} 가 KS={ks} 에 없다 — 정합 키를 만들 수 없다")
    return np.stack(F, axis=1), NM, b_lb

def dist(B, ref):
    d = B - ref[None]
    return np.sqrt((d * d).sum(axis=(1, 2)))

def cv_logit(X, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            mu = X[tr].mean(0); sd = X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X[tr] - mu) / sd, tt[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def two_template_cv(B, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            medN = np.median(B[tr & ~tt], axis=0); medS = np.median(B[tr & tt], axis=0)
            sc[te] = dist(B[te], medN) - dist(B[te], medS)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def run_struct(t_):
    prev_ = np.r_[False, t_[:-1]]; next_ = np.r_[t_[1:], False]
    iso = t_ & ~prev_ & ~next_
    rl = mx = 0
    for v in t_:
        rl = rl + 1 if v else 0
        mx = max(mx, rl)
    return float(iso.sum() / max(t_.sum(), 1)), int(mx)

def build_scores(X, NAMES, tt, MORPH, seed):
    """모든 팔의 비트별 점수. **라벨셔플 null 도 같은 함수로** 만든다(공정 · R26 ②)."""
    S = {nm_: X[:, j] for j, nm_ in enumerate(NAMES)}
    i3, i4, i5 = NAMES.index("f3"), NAMES.index("f4"), NAMES.index("f5")
    arms = {"lr_all": X, "lr_norr": X[:, [i3, i4]], "lr_norr5": X[:, [i3, i4, i5]],
            "lr_f1": X[:, [NAMES.index("f1")]]}
    for nm_, XX in arms.items():
        sc_ = cv_logit(XX, tt, K_FOLD, seed, N_REPEAT)
        if sc_ is None:
            return None
        S[nm_] = sc_
    for nm_, B_ in MORPH.items():
        st = two_template_cv(B_, tt, K_FOLD, seed, N_REPEAT)
        S[nm_] = st if st is not None else np.full(len(tt), np.nan)
    return S

run.log("\n" + "=" * 100)
run.log("【L-A】 특징·점수 계산 (형태 창 2종 · 국소 기저선 반환)")
run.log("=" * 100)
SCORES, META, SKIP = {}, {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if int(tt.sum()) < MIN_S_TPL or int((~tt).sum()) < MIN_S_TPL:
        SKIP.append((int(r), f"S {int(tt.sum())} · N {int((~tt).sum())}")); continue
    pre_m, post_m = PRE[mm], POST[mm]
    X, NAMES, lb_m = rhythm_feats(pre_m, post_m, KS)
    MORPH = {k: WIN[k][mm] for k in SEGS}
    S = build_scores(X, NAMES, tt, MORPH, SEED0)
    if S is None:
        SKIP.append((int(r), "겹 안 클래스 부족")); continue
    iso_f, mx_run = run_struct(tt)
    SCORES[int(r)] = S
    META[int(r)] = dict(tt=tt, pre=pre_m, post=post_m, lb=lb_m, X=X, MORPH=MORPH,
                        pos=int(tt.sum()), prev=float(tt.mean()),
                        iso_frac=iso_f, max_run=mx_run)
RS = sorted(SCORES)
run.log(f"  채점 {len(RS)}개체 · 제외 {len(SKIP)}개체")
if len(RS) < 10:
    raise AssetError("채점된 개체가 너무 적다")
ARMS = (["f1", "f2_8", "f2_16", "f3", "f4", "f5", "f6",
         "lr_f1", "lr_all", "lr_norr", "lr_norr5"] + list(SEGS))
MORPH_ARMS = list(SEGS)
RAW = {a: np.array([roc_auc_score(META[r]["tt"].astype(int), SCORES[r][a])
                    if np.isfinite(SCORES[r][a]).all() else np.nan for r in RS]) for a in ARMS}
NAMES_G = NAMES
run.log("  무정합 매크로 — " + " · ".join(f"{a} {np.nanmean(RAW[a]):.4f}" for a in
        ("f1", "f2_16", "f3", "stt", "p_late", "lr_all")))
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【L-B】 ★ 합동 정합 — `pre_rr` **값** × 국소 기저선 **대역**
# Q7-I‴: pre_rr 만 맞추면 f1 은 0.5 지만 f2 는 0.6688 로 산다(같은 절대 RR 이라도
# S 는 국소 기저선이 긴 자리에서 나오니까). 두 축을 **같이** 맞춰야 조기성이 통제된다.
def joint_key(pre_v, lb_v, bw):
    """**두 축**(조기성의 절대 눈금 · 국소 기저선)을 같은 폭으로 갈라 하나의 키로.

    `bw=None` 이면 **정확 pre 만**(Q7-I‴ 재현 — 곡선의 참조점). ★ 정확 pre 에 두 번째
    축을 얹으면 칸이 잘게 쪼개져 남는 게 없다 — 그래서 **두 축 모두 대역**으로 가고,
    대신 **짝 단위 잔여를 f1·f2 둘 다 실측**해 「정말 통제됐나」를 값으로 확인한다."""
    if bw is None:
        _, ia = np.unique(np.round(pre_v, 6), return_inverse=True)
        return ia.astype(np.int64)
    a = np.floor(pre_v / max(bw, 1e-9)).astype(np.int64); a = a - a.min()
    b = np.floor(lb_v / max(bw, 1e-9)).astype(np.int64); b = b - b.min()
    return a * (int(b.max()) + 1) + b

def cell_auc(sc, tt, key, min_s, min_n, pre_v=None, f2_v=None):
    """`key` 가 같은 비트끼리만 쌍을 센다. **평가만 제한**(승계).
    반환 (조건부 AUROC, 남은 S, 쌍, 칸수, **짝 단위 f1 잔여**, **짝 단위 f2 잔여**)."""
    uq, inv = np.unique(key, return_inverse=True)
    num = den = 0.0; ks_ = nc_ = 0; g1 = g2 = 0.0
    for j in range(len(uq)):
        m = inv == j
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if len(s_) < min_s or len(n_) < min_n:
            continue
        ks_ += len(s_); nc_ += 1
        gt = float((s_[:, None] > n_[None, :]).sum())
        eq = float((s_[:, None] == n_[None, :]).sum())
        num += gt + 0.5 * eq; den += float(len(s_) * len(n_))
        if pre_v is not None:
            a_, b_ = pre_v[m & tt], pre_v[m & ~tt]
            g1 += float(np.abs(a_[:, None] - b_[None, :]).sum())
            a2, b2 = f2_v[m & tt], f2_v[m & ~tt]
            g2 += float(np.abs(a2[:, None] - b2[None, :]).sum())
    if den < 1:
        return float("nan"), ks_, 0.0, nc_, float("nan"), float("nan")
    if pre_v is None:
        return num / den, ks_, den, nc_, float("nan"), float("nan")
    return (num / den, ks_, den, nc_,
            (g1 / den) / max(float(np.median(pre_v)), 1e-9), g2 / den)

run.log("\n" + "=" * 100)
run.log("【L-B】 합동 정합 곡선 — 두 축 같은 폭 · 짝 단위 잔여를 **f1·f2 둘 다**")
run.log("=" * 100)
CURVE = {}
for bw_f in (None,) + LB_BANDS:
    M = {a: np.full(len(RS), np.nan) for a in ARMS}
    ok = np.zeros(len(RS), bool)
    g1 = np.full(len(RS), np.nan); g2 = np.full(len(RS), np.nan)
    sf = np.full(len(RS), np.nan); nc = np.full(len(RS), np.nan)
    for i, r in enumerate(RS):
        tt = META[r]["tt"]; pre_m = META[r]["pre"]; lb_m = META[r]["lb"]
        f2_m = SCORES[r][f"f2_{LB_K}"]
        bw = None if bw_f is None else bw_f * float(np.median(pre_m))
        key = joint_key(pre_m, lb_m, bw)
        for a in ARMS:
            sc_ = SCORES[r][a]
            if not np.isfinite(sc_).all():
                continue
            v, ks_, pr_, nc_, r1, r2 = cell_auc(
                sc_, tt, key, MIN_CELL_S, MIN_CELL_N,
                pre_v=pre_m if a == "f1" else None, f2_v=f2_m if a == "f1" else None)
            M[a][i] = v
            if a == "f1":
                ok[i] = bool(ks_ >= MIN_JOINT_S and pr_ >= MIN_JOINT_PAIR)
                g1[i] = r1; g2[i] = r2; sf[i] = ks_ / max(META[r]["pos"], 1); nc[i] = nc_
    CURVE[bw_f] = dict(M=M, ok=ok, g1=g1, g2=g2, sf=sf, nc=nc)
    lab = "정확pre" if bw_f is None else f"{bw_f:.2f}"
    run.log(f"  두 축 대역 {lab:>7} — 가능 **{int(ok.sum()):>2}/{len(RS)}** · "
            f"남은 S {np.nanmedian(sf[ok]) if ok.any() else float('nan'):.3f} · "
            f"짝잔여 f1 **{np.nanmedian(g1[ok]) if ok.any() else float('nan'):.6f}** · "
            f"**f2 {np.nanmedian(g2[ok]) if ok.any() else float('nan'):.4f}** | "
            + " · ".join(f"{a} {np.nanmean(M[a][ok]) if ok.any() else float('nan'):.4f}"
                         for a in ("f1", "f2_16", "f3", "stt", "stt_safe")))

# ── ★ 유효 대역은 **규칙이 고른다** — 데이터를 보고 고르지 않는다
elig = [b for b in LB_BANDS if int(CURVE[b]["ok"].sum()) >= MIN_JOINT_REC]
if not elig:
    raise AssetError(f"합동 정합 가능 개체 ≥ {MIN_JOINT_REC} 인 대역이 없다 — 여기서 멈춘다")
BW = min(elig)
JOK = CURVE[BW]["ok"]; JM = CURVE[BW]["M"]
NJ = int(JOK.sum())
run.log(f"\n  ★ 유효 대역(규칙: 가능 ≥ {MIN_JOINT_REC} 중 가장 좁은 것) = **{BW:.2f}** · "
        f"개체 {NJ} · 칸수 중앙 {np.nanmedian(CURVE[BW]['nc'][JOK]):.0f}")
run.log(f"    (곡선 첫 줄 = Q7-I‴ 재현: 정확 pre 만 → f1 0.5000 인데 **f2 0.6688** 로 살았다)")
CONFIG["curve"] = {("pre_only" if b is None else f"{b:.2f}"): dict(
    n_ok=int(CURVE[b]["ok"].sum()),
    gap_f1=float(np.nanmedian(CURVE[b]["g1"][CURVE[b]["ok"]])) if CURVE[b]["ok"].any() else None,
    gap_f2=float(np.nanmedian(CURVE[b]["g2"][CURVE[b]["ok"]])) if CURVE[b]["ok"].any() else None,
    macro={a: float(np.nanmean(CURVE[b]["M"][a][CURVE[b]["ok"]])) if CURVE[b]["ok"].any()
           else None for a in ARMS}) for b in CURVE}
CONFIG["band_selected"] = BW
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【L-C】 합동 정합 조건 **라벨셔플 null** (R26 ②)
# 순수 특징(f1·f2·f3)의 영분포는 0.5 다. 학습이 낀 팔(lr_* · stt · stt_safe)은
# Q7-I‴ 에서 lr_f1 실측 0.4123 · null 0.4256 이었다 — **0.5 가 아니다.**
run.log("\n" + "=" * 100)
run.log(f"【L-C】 합동 정합 라벨셔플 null (셔플 {N_SHUF}회 · 대역 {BW:.2f})")
run.log("=" * 100)
NULL = {a: np.full(len(RS), np.nan) for a in ARMS}
NSE  = {a: np.full(len(RS), np.nan) for a in ARMS}   # ★ null 자체의 셔플 간 표준오차
for i, r in enumerate(RS):
    if not JOK[i]:
        continue
    tt = META[r]["tt"]; pre_m = META[r]["pre"]; lb_m = META[r]["lb"]
    key = joint_key(pre_m, lb_m, BW * float(np.median(pre_m)))
    acc = {a: [] for a in ARMS}
    for s_ in range(N_SHUF):
        rng = np.random.RandomState(SEED0 + 7919 * (s_ + 1) + int(r))
        ts = rng.permutation(tt)                       # 유병률 보존
        Ss = build_scores(META[r]["X"], NAMES_G, ts, META[r]["MORPH"], SEED0 + 31 * (s_ + 1))
        if Ss is None:
            continue
        for a in ARMS:
            sc_ = Ss[a]
            if not np.isfinite(sc_).all():
                continue
            v, _, pr_, _, _, _ = cell_auc(sc_, ts, key, MIN_CELL_S, MIN_CELL_N)
            if pr_ >= 1:
                acc[a].append(v)
    for a in ARMS:
        if len(acc[a]) >= 2:
            v_ = np.asarray(acc[a], float)
            NULL[a][i] = float(v_.mean())
            NSE[a][i] = float(v_.std(ddof=1) / np.sqrt(len(v_)))
        elif acc[a]:
            NULL[a][i] = float(acc[a][0]); NSE[a][i] = float("nan")
run.log(f"  팔별 — 실측 vs **셔플 null(±SE)** vs 초과분   [셔플 {N_SHUF}회]")
for a in ARMS:
    m_ = np.nanmean(JM[a][JOK]); n_ = np.nanmean(NULL[a][JOK])
    se_ = np.nanmean(NSE[a][JOK])
    star = "  ★" if a in MORPH_ARMS + ["f3"] else ""
    run.log(f"    {a:<9} {m_:.4f}  null {n_:.4f} ±{se_:.4f}  **초과 {m_-n_:+.4f}**{star}")
run.log("  ▸ null 의 SE 가 초과분과 비슷한 크기면 그 팔은 **셔플 횟수가 부족**한 것이다 —")
run.log("    아래 관문은 그 오차를 CI 에 **전파**해서 판정한다")
CONFIG["null"] = {a: float(np.nanmean(NULL[a][JOK])) for a in ARMS}
CONFIG["null_se"] = {a: float(np.nanmean(NSE[a][JOK])) for a in ARMS}
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【L-D】 관문 L1~L6
def boot_diff(a, b, seed, nb=NB_BOOT, mask=None, const=None, q=2.5):
    a = np.asarray(a, float)
    d = (a - const) if const is not None else (a - np.asarray(b, float))
    if mask is not None:
        d = d[mask]
    d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)])
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

def boot_excess(obs, nmean, nse, seed, mask, q=2.5, nb=NB_BOOT):
    """초과분(obs − null)의 개체 부트스트랩 + **null 자체의 셔플 오차를 함께 흔든다**.
    null 을 점추정으로 쓰면 CI 가 좁아진다(R22 ② 의 정신)."""
    d = (np.asarray(obs, float) - np.asarray(nmean, float))[mask]
    se = np.asarray(nse, float)[mask]
    ok = np.isfinite(d)
    d = d[ok]; se = np.where(np.isfinite(se[ok]), se[ok], 0.0)
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.empty(nb)
    for b in range(nb):
        idx = rng.randint(0, len(d), len(d))
        v[b] = (d[idx] - rng.normal(0.0, 1.0, len(idx)) * se[idx]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

run.log("\n" + "=" * 100)
run.log(f"【L-D】 관문 (합동 정합 · 대역 {BW:.2f} · 개체 {NJ})")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

# ── L1 절대 조기성이 죽었나 (정확 pre 참조점은 【L-B】 첫 줄)
m1, lo1, hi1, n1_ = boot_diff(JM["f1"], None, SEED0 + 1, mask=JOK, const=0.0)
DIFF["L1"] = dict(mean=m1, lo=lo1, hi=hi1, n=n1_)
g_("L1", decide(lo1, hi1, F1_COLLAPSE, "<"),
   f"f1ₘ **{m1:.4f}** [{lo1:.4f}, {hi1:.4f}] vs 상한 {F1_COLLAPSE}"
   f"   ← 절대 조기성 (곡선 첫 줄의 정확 pre 정합은 {EXACT_TIE} 미만이어야 한다)")

# ── L2 ★ 여기가 이 실험의 전제다
m2, lo2, hi2, n2_ = boot_diff(JM["f2_16"], None, SEED0 + 2, mask=JOK, const=0.0)
DIFF["L2"] = dict(mean=m2, lo=lo2, hi=hi2, n=n2_)
g_("L2", decide(lo2, hi2, F2_COLLAPSE, "<"),
   f"★ f2_16ₘ **{m2:.4f}** [{lo2:.4f}, {hi2:.4f}] vs 상한 {F2_COLLAPSE}"
   f"   ← **지지여야 아래를 「조기성 통제 후」로 읽는다** (Q7-I‴ 은 0.6688)")
SEALED = not VERD["L2"].startswith("✅")
if SEALED:
    run.log("    ⛔ **상대 조기성이 아직 안 죽었다.** L3~L6 값은 잔여 조기성을 담을 수")
    run.log("       있으므로 **「형태·post_rr 의 독립 기여」로 인용하지 않는다**(R24-b)")

# ── L3 ★ 형태의 진짜 독립 기여
exc_stt = JM["stt"] - NULL["stt"]
m3, lo3, hi3, n3_ = boot_excess(JM["stt"], NULL["stt"], NSE["stt"], SEED0 + 3,
                                JOK, q=BONF3 * 100)
DIFF["L3"] = dict(mean=m3, lo=lo3, hi=hi3, n=n3_)
g_("L3", decide(lo3, hi3, 0.0, ">"),
   f"★ STTₘ − nullSTTₘ **{m3:+.4f}** [{lo3:+.4f}, {hi3:+.4f}] · Bonferroni 3 · n={n3_}"
   f"   ← **형태의 진짜 독립 기여** (Q7-I‴ 은 +0.1380)")

# ── L4 post_rr 의 진짜 독립 기여 (순수 특징이라 null 은 0.5)
m4, lo4, hi4, n4_ = boot_diff(JM["f3"], None, SEED0 + 4, mask=JOK, const=0.5, q=BONF3 * 100)
DIFF["L4"] = dict(mean=m4, lo=lo4, hi=hi4, n=n4_)
g_("L4", decide(lo4, hi4, 0.0, ">"),
   f"f3ₘ − 0.5 **{m4:+.4f}** [{lo4:+.4f}, {hi4:+.4f}] · Bonferroni 3 · n={n4_}"
   f"   ← **post_rr 의 진짜 독립 기여** (Q7-I‴ 은 +0.0887)")

# ── L5 둘 중 누가 큰가
m5, lo5, hi5, n5_ = boot_diff(exc_stt, JM["f3"] - 0.5, SEED0 + 5, mask=JOK, q=BONF3 * 100)
DIFF["L5"] = dict(mean=m5, lo=lo5, hi=hi5, n=n5_)
g_("L5", decide(lo5, hi5, 0.0, ">"),
   f"(STT 초과) − (f3 초과) **{m5:+.4f}** [{lo5:+.4f}, {hi5:+.4f}] · Bonferroni 3")

# ── L6 (민감도 · 미보정) 다음 QRS 침입을 뺀 짧은 창
m6, lo6, hi6, n6_ = boot_excess(JM["stt_safe"], NULL["stt_safe"], NSE["stt_safe"],
                                SEED0 + 6, JOK)
DIFF["L6"] = dict(mean=m6, lo=lo6, hi=hi6, n=n6_)
g_("L6", decide(lo6, hi6, 0.0, ">"),
   f"(민감도·미보정) STT_SAFEₘ − nullₘ **{m6:+.4f}** [{lo6:+.4f}, {hi6:+.4f}] · 창 {STT_SAFE}"
   f"   ← 침입 불가 폭에서도 형태가 남나")

# ── L7 ★ Q7-F 의 P 창을 **오염되지 않은 조건에서 처음** 잰다
m7, lo7, hi7, n7_ = boot_excess(JM["p_late"], NULL["p_late"], NSE["p_late"],
                                SEED0 + 7, JOK, q=BONF3 * 100)
DIFF["L7"] = dict(mean=m7, lo=lo7, hi=hi7, n=n7_)
g_("L7", decide(lo7, hi7, 0.0, ">"),
   f"★ P_lateₘ − nullₘ **{m7:+.4f}** [{lo7:+.4f}, {hi7:+.4f}] · Bonferroni 3 · n={n7_}"
   f"   ← **Q7-F 의 「형태 상한 0.6890」은 절대 RR 정합값이라 무효**(K3). 여기가 진짜다")
for nm_ in ("p_full", "p_early"):
    mx_, lx_, hx_, _ = boot_excess(JM[nm_], NULL[nm_], NSE[nm_], SEED0 + 70, JOK)
    run.log(f"    (참고·미보정) {nm_}ₘ − nullₘ {mx_:+.4f} [{lx_:+.4f}, {hx_:+.4f}]")
mcmp, lcmp, hcmp, _ = boot_diff(JM["p_late"] - NULL["p_late"],
                                JM["stt"] - NULL["stt"], SEED0 + 71, mask=JOK)
run.log(f"    (참고·미보정) P_late 초과 − STT 초과 {mcmp:+.4f} [{lcmp:+.4f}, {hcmp:+.4f}]"
        f"   ← 비슷하면 **「P 파 고유의 신호는 없다」**가 확정된다")

# ── L8 (민감도 · 미보정) post_rr 까지 정합하면 STT 가 사라지나
#     재분극은 rate hysteresis 가 있어 post_rr 로도 형태가 샐 수 있다(외부 지적).
POSTB = BW * 2.0        # ★ 3축이라 칸이 잘게 쪼개진다 — post 축은 두 배 폭으로 사전등록
j8 = np.full(len(RS), np.nan); n8 = np.full(len(RS), np.nan); o8 = np.zeros(len(RS), bool)
for i, r in enumerate(RS):
    if not JOK[i]:
        continue
    tt = META[r]["tt"]; pre_m = META[r]["pre"]; med_ = float(np.median(pre_m))
    k2 = joint_key(pre_m, META[r]["lb"], BW * med_)
    pb = np.floor(META[r]["post"] / max(POSTB * med_, 1e-9)).astype(np.int64)
    pb = pb - pb.min()
    key3 = k2 * (int(pb.max()) + 1) + pb
    v, ks_, pr_, _, _, _ = cell_auc(SCORES[r]["stt"], tt, key3, MIN_CELL_S, MIN_CELL_N)
    if pr_ >= MIN_JOINT_PAIR and ks_ >= MIN_JOINT_S:
        j8[i] = v; o8[i] = True
        vs = []
        for s_ in range(min(N_SHUF, 8)):
            rng = np.random.RandomState(SEED0 + 613 * (s_ + 1) + int(r))
            ts = rng.permutation(tt)
            Ss = build_scores(META[r]["X"], NAMES_G, ts, {"stt": META[r]["MORPH"]["stt"]},
                              SEED0 + 17 * (s_ + 1))
            if Ss is None:
                continue
            vv, _, pr2, _, _, _ = cell_auc(Ss["stt"], ts, key3, MIN_CELL_S, MIN_CELL_N)
            if pr2 >= 1:
                vs.append(vv)
        if vs:
            n8[i] = float(np.mean(vs))
m8, lo8, hi8, n8_ = boot_diff(j8, n8, SEED0 + 8, mask=o8)
DIFF["L8"] = dict(mean=m8, lo=lo8, hi=hi8, n=n8_, post_band=POSTB)
g_("L8", decide(lo8, hi8, 0.0, ">"),
   f"(민감도·미보정) **3축 정합**(pre 값 × 국소기저선 {BW:.2f} × post {POSTB:.2f}) STT 초과 "
   f"**{m8:+.4f}** [{lo8:+.4f}, {hi8:+.4f}] · 개체 {int(o8.sum())}"
   f"   ← 사라지면 STT 는 **재분극의 rate hysteresis** 경로였다")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF; CONFIG["sealed"] = bool(SEALED)
CONFIG["n_post_matched"] = int(o8.sum())
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【L-E】 L7 — 검정력과 교란을 숫자로 (관문 아님)
run.log("\n" + "=" * 100)
run.log("【L-E】 L9 — ★ 선택 편향 · 계수 안정성 · 침입률 · 층화 일치도")
run.log("=" * 100)

# ── ⓪ ★ 정합 코호트의 선택 편향 — 「누가 남았나」를 먼저 본다
#     합동 정합이 가능한 개체는 **S 와 N 의 RR 분포가 겹치는** 개체다. 즉 조기성이
#     덜 뚜렷한 쪽이 남는다. 층 구성을 안 보고 결론을 일반화하면 안 된다.
ISOF = np.array([META[r]["iso_frac"] for r in RS])
LAY = (("고립 S", ISOF >= ISO_HI), ("혼합", (ISOF < ISO_HI) & (ISOF > ISO_LO)),
       ("런 우세", ISOF <= ISO_LO))
PREV = np.array([META[r]["prev"] for r in RS])
RRSEP = np.array([abs(np.median(META[r]["pre"][~META[r]["tt"]])
                      - np.median(META[r]["pre"][META[r]["tt"]]))
                  / max(np.median(META[r]["pre"]), 1e-9) for r in RS])
run.log(f"  합동 정합 가능 **{NJ}/{len(RS)}** 개체 — 층 구성")
for nm_, msk in LAY:
    tot = int(msk.sum()); got = int((msk & JOK).sum())
    run.log(f"    {nm_:<8} 전체 {tot:>2} → 정합 가능 **{got:>2}**"
            + ("   ⛔ **한 개도 안 남았다**" if tot > 0 and got == 0 else ""))
run.log(f"  RR 분리도 — 정합 가능 {np.nanmedian(RRSEP[JOK]):.4f} vs "
        f"제외 {np.nanmedian(RRSEP[~JOK]) if (~JOK).any() else float('nan'):.4f}"
        f"   ← 정합 가능 쪽이 작으면 **조기성이 덜 뚜렷한 개체만 남은 것**")
run.log(f"  유병률   — 정합 가능 {np.nanmedian(PREV[JOK]):.4f} vs "
        f"제외 {np.nanmedian(PREV[~JOK]) if (~JOK).any() else float('nan'):.4f}")
run.log("  ▸ 특정 층이 0 이면 **결론의 적용 범위를 그 층 밖으로 넓히지 않는다**(사전등록)")
CONFIG["selection"] = dict(
    layers={nm_: dict(total=int(m.sum()), matched=int((m & JOK).sum())) for nm_, m in LAY},
    rr_sep_matched=float(np.nanmedian(RRSEP[JOK])),
    rr_sep_excluded=float(np.nanmedian(RRSEP[~JOK])) if (~JOK).any() else None,
    prev_matched=float(np.nanmedian(PREV[JOK])))

# ── ① ★ 억제 변수 경고 — f5·f6 은 단독 부호가 반대다(Q7-I‴: −0.080 · −0.167).
#     조합에서만 값을 내는 특징은 **계수 부호가 표본에 따라 흔들린다.**
run.log("\n  폴드 간 계수 부호 안정성 (LR(전부) · 겹 25회 = 5겹 × 5반복 재적합)")
SIGN = {nm_: [] for nm_ in NAMES_G}
for i, r in enumerate(RS):
    if not JOK[i]:
        continue
    X_ = META[r]["X"]; tt = META[r]["tt"]
    co = []
    for rep in range(5):
        rng = np.random.RandomState(SEED0 + 991 * rep + int(r))
        fold = rng.permutation(len(tt)) % K_FOLD
        for f in range(K_FOLD):
            tr = fold != f
            if int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                continue
            mu = X_[tr].mean(0); sd = X_[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X_[tr] - mu) / sd, tt[tr].astype(int))
            co.append(lr.coef_[0])
    if co:
        co = np.stack(co)
        for j, nm_ in enumerate(NAMES_G):
            SIGN[nm_].append(float(max((co[:, j] > 0).mean(), (co[:, j] < 0).mean())))
for nm_ in NAMES_G:
    v_ = np.array(SIGN[nm_], float)
    flag = "   ⚠️ **부호가 흔들린다**" if np.nanmean(v_) < 0.85 else ""
    run.log(f"    {nm_:<7} 부호 일치율 중앙 {np.nanmedian(v_):.3f} · "
            f"평균 {np.nanmean(v_):.3f}{flag}")
run.log("  ▸ 억제 변수(단독 AUROC < 0.5 인데 조합에서 기여)는 일치율이 낮게 나오는 게")
run.log("    정상이지만, **낮으면 그 특징의 기여를 개체 밖으로 일반화하지 않는다**")
CONFIG["coef_sign"] = {nm_: float(np.nanmean(SIGN[nm_])) for nm_ in NAMES_G}

# ── ① 침입률: post_rr 이 짧으면 다음 QRS 가 STT 창에 든다 (STT_SAFE 는 불가)
run.log(f"  침입 기준 — post_rr < {INTRUDE_LIM} 샘플 ({INTRUDE_LIM/360*1000:.0f}ms) 이면 "
        f"다음 QRS 가 STT{STT_SEG} 에 든다")
ints, intn = [], []
for r in RS:
    tt = META[r]["tt"]; po = META[r]["post"]
    ints.append(float((po[tt] < INTRUDE_LIM).mean()))
    intn.append(float((po[~tt] < INTRUDE_LIM).mean()))
ints, intn = np.array(ints), np.array(intn)
run.log(f"  전체 {len(RS)}개체 — S {np.nanmean(ints):.4f} · N {np.nanmean(intn):.4f} · "
        f"차이 {np.nanmean(ints - intn):+.4f}")
for nm_, msk in LAY:
    if not msk.any():
        continue
    run.log(f"    {nm_:<8} {int(msk.sum()):>2}개체 · S {np.nanmean(ints[msk]):.4f} · "
            f"N {np.nanmean(intn[msk]):.4f}")
run.log(f"  ▸ STT_SAFE{STT_SAFE} 는 침입하려면 post_rr < {STT_SAFE[1]-100} 샘플"
        f"({(STT_SAFE[1]-100)/360*1000:.0f}ms · {60/((STT_SAFE[1]-100)/360):.0f}bpm) 이라 사실상 불가")

# ── ② 비트 수준 층화 일치도 — 매크로가 버리는 검정력을 되찾는다(보조 지표)
run.log("\n  비트 수준 층화 일치도 (칸을 층으로 · 개체 무가중 · **보조 지표**)")
POOL = {}
for a in ("f1", "f2_16", "f3", "stt", "stt_safe", "p_late", "lr_all"):
    num = den = 0.0
    for i, r in enumerate(RS):
        if not JOK[i]:
            continue
        tt = META[r]["tt"]; pre_m = META[r]["pre"]
        key = joint_key(pre_m, META[r]["lb"], BW * float(np.median(pre_m)))
        sc_ = SCORES[r][a]
        if not np.isfinite(sc_).all():
            continue
        v, _, pr_, _, _, _ = cell_auc(sc_, tt, key, MIN_CELL_S, MIN_CELL_N)
        if pr_ >= 1:
            num += v * pr_; den += pr_
    POOL[a] = num / den if den > 0 else float("nan")
    run.log(f"    {a:<9} 층화 {POOL[a]:.4f}  (매크로 {np.nanmean(JM[a][JOK]):.4f})")
run.log("  ▸ 둘이 크게 갈리면 **큰 개체가 방향을 끌고 있다**는 뜻이다 — 매크로가 주지표다")
CONFIG["pooled"] = POOL
CONFIG["intrusion"] = dict(s=float(np.nanmean(ints)), n=float(np.nanmean(intn)))
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【L-F】 그림 · 관문 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다(네모 방지).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

XS = [0.012 if b is None else b for b in CURVE]      # pre 만 = 곡선 왼쪽 밖
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))
for a_, c_, lb in (("f1", "tab:red", "f1 (must be 0.5)"),
                   ("f2_16", "tab:purple", "f2_16 (must fall)"),
                   ("f3", "tab:green", "f3"), ("stt", "tab:orange", "STT"),
                   ("p_late", "tab:blue", "P_late")):
    ax[0].plot(XS, [np.nanmean(CURVE[b]["M"][a_][CURVE[b]["ok"]]) for b in CURVE],
               "o-", color=c_, label=lb)
ax[0].axhline(0.5, ls="--", lw=0.8, color="crimson")
ax[0].axhline(F2_COLLAPSE, ls=":", lw=0.8, color="purple")
ax[0].set_xlabel("local-baseline band  (0.012 = pre-only, Q7-I3)  <- narrower")
ax[0].set_ylabel("jointly matched macro AUROC")
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

ax[1].plot(XS, [np.nanmedian(CURVE[b]["g2"][CURVE[b]["ok"]]) for b in CURVE],
           "s-", color="purple", label="f2 residual")
ax[1].plot(XS, [np.nanmedian(CURVE[b]["g1"][CURVE[b]["ok"]]) for b in CURVE],
           "s-", color="crimson", label="f1 residual")
ax[1].set_yscale("symlog", linthresh=1e-4)
ax[1].set_xlabel("local-baseline band  <- narrower")
ax[1].set_ylabel("PAIRWISE residual"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)

aa = [a for a in ARMS if a != "f2_8"]
ex_ = [np.nanmean(JM[a][JOK]) - np.nanmean(NULL[a][JOK]) for a in aa]
ax[2].barh(range(len(aa)), ex_, color=["tab:green" if v > 0 else "tab:red" for v in ex_])
ax[2].set_yticks(range(len(aa))); ax[2].set_yticklabels(aa, fontsize=7)
ax[2].axvline(0, color="k", lw=.8)
ax[2].set_xlabel("excess over own label-shuffle null (joint match)")
ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q7k_relative_match", fig)
plt.close(fig)
display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("관문 요약")
run.log("=" * 100)
for k in ("L1", "L2", "L3", "L4", "L5", "L6"):
    run.log(f"  {k:<4}{VERD.get(k, '(미실행)')}")
run.log("")
if SEALED:
    run.log("  ⛔ **L2 미결/기각 — 합동 정합으로도 상대 조기성이 안 죽었다.**")
    run.log("     L3~L6 을 「형태·post_rr 의 독립 기여」로 인용하지 않는다.")
    run.log("     다음 축(예: post_rr 정합 · f2 잔차화)을 더 걸어야 한다")
elif VERD.get("L3", "").startswith("✅"):
    run.log("  ★ **조기성을 둘 다 죽인 조건에서 형태가 남았다.**")
    run.log("     이 퀘스트에서 처음으로 **형태 축의 독립 기여**를 말할 수 있다 → Q7-F′·Q7-G")
elif VERD.get("L4", "").startswith("✅"):
    run.log("  ★ 형태는 안 남고 **post_rr 만 남았다** — S 는 끝까지 리듬 문제다")
else:
    run.log("  ⛔ **조기성을 다 빼면 아무 것도 안 남는다.** S 판별은 조기성 그 자체이고,")
    run.log("     형태·post_rr 는 독립 기여가 없다 — 이것도 결과다")

run.finish({
    "exp_id": "quest46_q7k_relative_match",
    "metric": "svdb_joint_match_stt_excess",
    "value": float(np.nanmean(JM["stt"][JOK]) - np.nanmean(NULL["stt"][JOK])),
    "passed": bool(VERD.get("L2", "").startswith("✅") and VERD.get("L3", "").startswith("✅")),
    "summary": ("pre_rr 값 × 국소 기저선 대역으로 합동 정합해 f1·f2 를 둘 다 통제한 뒤 "
                "형태(STT)와 post_rr(f3)의 독립 기여를 물었다."),
    "verdicts": VERD, "diffs": DIFF, "band_selected": BW, "sealed": bool(SEALED),
    "curve": CONFIG.get("curve", {}), "null": CONFIG.get("null", {}),
    "pooled": CONFIG.get("pooled", {}), "intrusion": CONFIG.get("intrusion", {}),
    "n_scored": len(RS), "n_joint": NJ, "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-relative-match`")